# Additional experiment: training-noise distribution

This notebook tests whether a direct one-stage coordinate model benefits from clean, noisy, or mixed training data. It preserves the official protocol: validation and test gradients are clean. The mixed condition uses the same sampled training shapes with one clean and one independently noisy gradient realization. Run manually; this notebook does not execute during repository checks.

In [ ]:
from pathlib import Path
import json
import sys
import gc
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import OneModelRunConfig, Stage2ModelConfig, StageTrainingConfig
from datasets import build_two_stage_datasets
from models import (evaluate_stage2_predictions, evaluate_stage2_predictions_by_shape,
                    select_best_stage2_threshold, set_torch_seed)
from models.one_model import GradientToMaskModel, fit_one_model, predict_one_model_logits

N = 10
SIGMA = 0.01
SEED = 42
TRAINING_SAMPLES = 20_000
MIXED_HALF_SAMPLES = TRAINING_SAMPLES // 2
VALIDATION_SAMPLES = 2_000
TEST_SAMPLES = 1_000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_ROOT = ROOT / 'outputs' / 'additional_one_stage_training_distribution'
NOTES_ROOT = ROOT / 'docs' / 'experiments' / 'sigma01_results'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NOTES_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
print('Device:', DEVICE)

In [ ]:
def make_config(name, sigma, training_samples):
    model = Stage2ModelConfig(
        hidden_layer_sizes=(512, 1024), dropout_rates=(0.1, 0.1),
        model_type='coord_conv_decoder', latent_grid_size=16, latent_channels=160,
        decoder_channels=(160, 128, 96, 64, 32), use_rectangle_edge_weighting=True,
        use_foreground_pos_weight=False, rectangle_edge_weight=4.0, rectangle_edge_width=3,
        edge_weight_mode='all', annulus_edge_weight=1.0, annulus_edge_width=3,
        training=StageTrainingConfig(
            epochs=170, batch_size=96, learning_rate=0.0005, validation_frequency=60,
            verbose=False, early_stopping_patience=25, min_epochs=50,
            min_improvement=0.001, lr_drop_factor=0.5, lr_drop_period=80,
            weight_decay=0.00025, gradient_clip_norm=0.8, loss_type='bce_dice',
            dice_loss_weight=1.0, dice_smooth=1.0,
        ),
    )
    return OneModelRunConfig(
        N=N, training_samples=training_samples, validation_samples=VALIDATION_SAMPLES,
        test_samples=TEST_SAMPLES, noise_sigma=sigma, noise_mode='absolute', seed=SEED,
        training_noise_replicas=1,
        training_shape_weights=(('rectangle', 0.25), ('two_circles', 0.45),
                                ('annulus', 0.10), ('ellipse', 0.15), ('circle', 0.05)),
        use_validation_threshold_sweep=True, model=model, output_dir=OUTPUT_ROOT / name,
    )

def training_options(config):
    t = config.model.training
    return {
        'epochs': t.epochs, 'batch_size': t.batch_size, 'learning_rate': t.learning_rate,
        'device': DEVICE, 'validation_frequency': t.validation_frequency, 'verbose': t.verbose,
        'early_stopping_patience': t.early_stopping_patience, 'min_epochs': t.min_epochs,
        'min_improvement': t.min_improvement, 'lr_drop_factor': t.lr_drop_factor,
        'lr_drop_period': t.lr_drop_period, 'weight_decay': t.weight_decay,
        'gradient_clip_norm': t.gradient_clip_norm, 'loss_type': t.loss_type,
        'dice_loss_weight': t.dice_loss_weight, 'dice_smooth': t.dice_smooth,
        'grid_size': config.grid_size, 'train_shape_types': None,
        'use_rectangle_edge_weighting': config.model.use_rectangle_edge_weighting,
        'rectangle_edge_weight': config.model.rectangle_edge_weight,
        'rectangle_edge_width': config.model.rectangle_edge_width,
        'edge_weight_mode': config.model.edge_weight_mode,
        'annulus_edge_weight': config.model.annulus_edge_weight,
        'annulus_edge_width': config.model.annulus_edge_width,
        'use_foreground_pos_weight': config.model.use_foreground_pos_weight,
    }

In [ ]:
# Build matched datasets. The clean and noisy datasets use the same seed, so their
# sampled shapes and clean validation/test sets are aligned.
clean_source = build_two_stage_datasets(make_config('clean_source', 0.0, TRAINING_SAMPLES))
noisy_source = build_two_stage_datasets(make_config('noisy_source', SIGMA, TRAINING_SAMPLES))

def train_arrays(condition, train_features, train_masks, train_shape_types, eval_bundle, config):
    set_torch_seed(SEED)
    model = GradientToMaskModel(
        input_dim=config.gradient_feature_size, output_dim=config.mask_pixels,
        hidden_dims=config.model.hidden_layer_sizes, dropout_rates=config.model.dropout_rates,
        latent_grid_size=config.model.latent_grid_size, latent_channels=config.model.latent_channels,
        decoder_channels=config.model.decoder_channels,
    )
    options = training_options(config)
    options['train_shape_types'] = train_shape_types
    training_result = fit_one_model(
        model=model, gradient_features=train_features, target_masks=train_masks,
        validation_gradient_features=eval_bundle.validation.gradient_data,
        validation_masks=eval_bundle.validation.masks, **options,
    )
    validation_logits = predict_one_model_logits(model, eval_bundle.validation.gradient_data, DEVICE, training_result)
    test_logits = predict_one_model_logits(model, eval_bundle.test.gradient_data, DEVICE, training_result)
    fixed_logits = predict_one_model_logits(model, eval_bundle.fixed.gradient_data, DEVICE, training_result)
    threshold_summary = select_best_stage2_threshold(eval_bundle.validation.masks, validation_logits, config.threshold_candidates)
    threshold = float(threshold_summary['selected_threshold'])
    summary = {
        'condition': condition, 'training_rows': int(train_features.shape[0]),
        'metrics': {
            'test': evaluate_stage2_predictions(eval_bundle.test.masks, test_logits, threshold),
            'fixed': evaluate_stage2_predictions(eval_bundle.fixed.masks, fixed_logits, threshold),
        },
        'metrics_by_shape': {
            'test': evaluate_stage2_predictions_by_shape(eval_bundle.test.masks, test_logits, threshold, eval_bundle.test.shape_types),
        },
        'threshold_summary': threshold_summary,
    }
    condition_root = OUTPUT_ROOT / condition
    condition_root.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), condition_root / 'one_model_weights.pt')
    (condition_root / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

In [ ]:
conditions = {
    'clean_training': (clean_source.train.gradient_data, clean_source.train.masks, clean_source.train.shape_types, clean_source),
    'sigma01_training': (noisy_source.train.gradient_data, noisy_source.train.masks, noisy_source.train.shape_types, noisy_source),
    'mixed_clean_sigma01': (
        np.concatenate([clean_source.train.gradient_data[:MIXED_HALF_SAMPLES], noisy_source.train.gradient_data[:MIXED_HALF_SAMPLES]], axis=0),
        np.concatenate([clean_source.train.masks[:MIXED_HALF_SAMPLES], noisy_source.train.masks[:MIXED_HALF_SAMPLES]], axis=0),
        clean_source.train.shape_types[:MIXED_HALF_SAMPLES] + noisy_source.train.shape_types[:MIXED_HALF_SAMPLES],
        clean_source,
    ),
}

results = []
shape_results = []
for condition, (features, masks, shape_types, eval_bundle) in conditions.items():
    print(f'\n=== ADDITIONAL ONE-STAGE CONDITION: {condition} ===', flush=True)
    config = make_config(condition, 0.0, features.shape[0])
    summary = train_arrays(condition, features, masks, shape_types, eval_bundle, config)
    row = {
        'condition': condition, 'training_rows': features.shape[0],
        'test_iou': summary['metrics']['test']['mean_iou'],
        'fixed_iou': summary['metrics']['fixed']['mean_iou'],
        'validation_iou': summary['threshold_summary']['validation_metrics']['mean_iou'],
        'threshold': summary['threshold_summary']['selected_threshold'],
    }
    results.append(row)
    for shape, metrics in summary['metrics_by_shape']['test'].items():
        shape_results.append({'condition': condition, 'shape': shape, **metrics})
    print(json.dumps(row, indent=2), flush=True)
    del summary, config
    gc.collect()

comparison = pd.DataFrame(results).sort_values('test_iou', ascending=False)
by_shape = pd.DataFrame(shape_results).sort_values(['condition', 'shape'])
comparison.to_csv(NOTES_ROOT / 'additional_one_stage_results.csv', index=False)
by_shape.to_csv(NOTES_ROOT / 'additional_one_stage_by_shape.csv', index=False)
(NOTES_ROOT / 'additional_one_stage_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('\n=== ADDITIONAL EXPERIMENT RESULTS ===')
print(comparison.to_string(index=False))
print('\n=== BY SHAPE ===')
print(by_shape.to_string(index=False))
comparison